# Audit data engineering NOURISH — 21 Oktober 2026
Companion rerunnable untuk pemeriksaan CSV lokal. Web tidak dijalankan. Hasil temuan tersimpan pada DATA_ENGINEERING_REVIEW.md; output notebook sengaja tidak dibuat-buat. Periksa nama wilayah secara semantik, bukan hanya perbedaan ejaan.


In [ ]:
from pathlib import Path
import csv, collections, re, json
root = Path(r"A:\0000_CurrentlyWorkingOn\ASEANDSE\##Data")
def read(name):
    with (root / name).open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))
def norm(s):
    return re.sub("[^A-Z]", "", re.sub(r"^(KABUPATEN|KAB\.?|KOTA)\s+", "", s.upper()))
master = read("20_indonesia_district_join_preview.csv")
print("Master rows / unique keys:", len(master), len({r["kab_code"] for r in master}))
for k in ["pou_pct", "population", "undernourished_people", "ikp", "pph", "energy_kcal_cap_day", "protein_g_cap_day"]:
    print(k, "valid", sum(bool(r[k]) for r in master))
pou = read("01_pou_kabkota_2025_clean.csv")
counts = collections.Counter((r["canonical_kab_code"], r["year"]) for r in pou)
print("PoU duplicated code-year keys:", sum(n > 1 for n in counts.values()))
pph = read("06_pph_kabkota_clean.csv")
candidates = [r for r in pph if r["canonical_kab_code"].startswith("9") and norm(r["source_kab_original"]) != norm(r["canonical_kab_kota"])]
print("Papua name mismatch candidates per year:", dict(collections.Counter(r["year"] for r in candidates)))
print("Review names to distinguish spelling aliases (Biak Namfor) from different districts.")
for r in candidates:
    if r["year"] == "2025":
        print(r["canonical_kab_code"], r["source_kab_original"], "=>", r["canonical_kab_kota"], r["pph"], r["match_confidence"])
rounding_fails = []
for r in master:
    if all(r[k] for k in ["pou_pct", "population", "undernourished_people"]):
        pop = float(r["population"])
        error = abs(float(r["undernourished_people"]) - float(r["pou_pct"]) / 100 * pop)
        if error > pop * 0.00005 + 1:
            rounding_fails.append(r["kab_code"])
print("Outside hypothesized 0.01-percentage-point rounding tolerance:", rounding_fails)
ranked = sorted(master, key=lambda r: -float(r["ikp"]))
print("IKP rank order mismatches:", sum(int(r["ikp_rank"]) != i + 1 for i, r in enumerate(ranked)))
for name in ["02_pou_kabkota_historical_clean.csv", "09_food_consumption_long_clean.csv", "10_skpg_province_clean.csv", "13_food_price_anomaly_national_clean.csv"]:
    rr = read(name)
    print(name, len(rr), dict(collections.Counter(r.get("year") for r in rr)))
